## DETree: DEtecting Human-AI Collaborative Texts
## End-to-End Paper Implementation — NeurIPS 2025
**Paper:** arXiv:2510.17489 | **GitHub:** heyongxin233/DETree

>  **Runtime:** Runtime → Change runtime type → **T4 GPU** before running

### What this notebook reproduces:
- Table 1: Supervised Detection (AvgRec + F1)
- Table 2: Adversarial Robustness
- Figure 2: t-SNE Embedding Space
- Figure 4: Bar Charts vs Paper
- Figure 5: k-Ablation
- Screenshot Detection Demo
- Gradio Interactive Demo


---
## Section 1 — GPU Check


In [ ]:
import torch
assert torch.cuda.is_available(), 'ERROR: No GPU! Go to Runtime → Change runtime type → T4 GPU'
print(' GPU:', torch.cuda.get_device_name(0))
print('   VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')


 GPU: Tesla T4
   VRAM: 15.6 GB


---
## Section 2 — Install & Clone


In [ ]:
!git clone https://github.com/jampala250897/DTREE_MS_JVK.git -q
!sed -i '/faiss-gpu/d' DTREE_MS_JVK/DETree-main/requirements.txt
!pip install -r DTREE_MS_JVK/DETree-main/requirements.txt -q
!pip install faiss-cpu scikit-learn matplotlib seaborn pandas tqdm Pillow pytesseract gradio -q
print(' All packages installed')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 113.6 MB/s eta 0:00:00
 All packages installed


---
## Section 3 — Login & Download Data


In [ ]:
from huggingface_hub import login
login()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:

from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="heyongxin233/RealBench",
    repo_type="dataset",

)

for f in files:
    print(f)

.gitattributes
Deepfake/extend/test.jsonl
Deepfake/extend/train.jsonl
Deepfake/extend/valid.jsonl
Deepfake/no_attack/test.jsonl
Deepfake/no_attack/test_ood.jsonl
Deepfake/no_attack/train.jsonl
Deepfake/no_attack/valid.jsonl
Deepfake/paraphrase_by_llm/test.jsonl
Deepfake/paraphrase_by_llm/train.jsonl
Deepfake/paraphrase_by_llm/valid.jsonl
Deepfake/perplexity_attack/test.jsonl
Deepfake/perplexity_attack/train.jsonl
Deepfake/perplexity_attack/valid.jsonl
Deepfake/polish/test.jsonl
Deepfake/polish/train.jsonl
Deepfake/polish/valid.jsonl
Deepfake/synonym/test.jsonl
Deepfake/synonym/train.jsonl
Deepfake/synonym/valid.jsonl
Deepfake/translate/de-en-test.jsonl
Deepfake/translate/de-en-train.jsonl
Deepfake/translate/de-en-valid.jsonl
Deepfake/translate/en-de/en-de-test.jsonl
Deepfake/translate/en-de/en-de-train.jsonl
Deepfake/translate/en-de/en-de-valid.jsonl
Deepfake/translate/en-zh/test.jsonl
Deepfake/translate/en-zh/train.jsonl
Deepfake/translate/en-zh/valid.jsonl
Deepfake/translate/test.jso

In [ ]:

from huggingface_hub import hf_hub_download
import os

os.makedirs('/content/RealBench/embbedings', exist_ok=True)


hf_hub_download(
    repo_id='heyongxin233/RealBench', repo_type='dataset',
    filename='embbedings/priori1_center10k.pt',
    local_dir='/content/RealBench'
)
hf_hub_download(
    repo_id='heyongxin233/RealBench', repo_type='dataset',
    filename='embbedings/mage_center10k.pt',
    local_dir='/content/RealBench'
)
print(' Embeddings downloaded')

# Dtf
FILES = [
    'MAGE/no_attack/test.jsonl',
    'M4_monolingual/no_attack/train.jsonl',
    'M4_multilingual/no_attack/train.jsonl',
    'TuringBench/no_attack/train.jsonl',
   # 'Deepfake/no_attack/test.jsonl',
   # 'OUTFOX/no_attack/test.jsonl',
   # 'Deepfake/paraphrase_by_llm/test.jsonl',
   # 'Deepfake/perplexity_attack/test.jsonl',
   # 'Deepfake/synonym/test.jsonl',
]

downloaded = {}
for f in FILES:
    try:
        local = hf_hub_download(
            repo_id='heyongxin233/RealBench', repo_type='dataset',
            filename=f, local_dir='/content/RealBench'
        )
        downloaded[f] = local
        print(f'   {f}')
    except Exception as e:
        print(f'   {f}: {e}')

print(f'\nDownloaded {len(downloaded)} files')


embbedings/priori1_center10k.pt:   0%|          | 0.00/164M [00:00<?, ?B/s]

embbedings/mage_center10k.pt:   0%|          | 0.00/205M [00:00<?, ?B/s]

 Embeddings downloaded
   MAGE/no_attack/test.jsonl: 404 Client Error. (Request ID: Root=1-69f4fd26-01c57be11aa3e37d4aabb803;a459f71b-4cb3-4c3b-a26b-70884ffec270)

Entry Not Found for url: https://huggingface.co/datasets/heyongxin233/RealBench/resolve/main/MAGE/no_attack/test.jsonl.


M4_monolingual/no_attack/train.jsonl:   0%|          | 0.00/346M [00:00<?, ?B/s]

   M4_monolingual/no_attack/train.jsonl


M4_multilingual/no_attack/train.jsonl:   0%|          | 0.00/590M [00:00<?, ?B/s]

   M4_multilingual/no_attack/train.jsonl


TuringBench/no_attack/train.jsonl:   0%|          | 0.00/148M [00:00<?, ?B/s]

   TuringBench/no_attack/train.jsonl

Downloaded 3 files


In [ ]:

import torch
import numpy as np

priori_path = '/content/RealBench/embbedings/priori1_center10k.pt'
mage_path   = '/content/RealBench/embbedings/mage_center10k.pt'

priori_data = torch.load(priori_path, map_location='cpu')
mage_data   = torch.load(mage_path, map_location='cpu')

priori_labels = np.array(priori_data['labels'])
mage_labels   = np.array(mage_data['labels'])

print("priori classes:", priori_data['classes'])
print("mage classes:", mage_data['classes'])

print("\npriori unique labels:", np.unique(priori_labels))
print("mage unique labels:", np.unique(mage_labels))

print("\npriori counts:")
for v in np.unique(priori_labels):
    print(f"label {v}: {(priori_labels == v).sum()}")

print("\nmage counts:")
for v in np.unique(mage_labels):
    print(f"label {v}: {(mage_labels == v).sum()}")

classes_match = list(priori_data['classes']) == list(mage_data['classes'])
label_set_match = np.array_equal(np.unique(priori_labels), np.unique(mage_labels))

print("\nClasses match:", classes_match)
print("Unique label set matches:", label_set_match)

if classes_match and label_set_match:
    print(" Both DBs appear to have compatible label format")
else:
    print(" Check class ordering or label encoding before merging")

priori classes: ['llm', 'human']
mage classes: ['llm', 'human']

priori unique labels: [0 1]
mage unique labels: [0 1]

priori counts:
label 0: 10000
label 1: 10000

mage counts:
label 0: 10000
label 1: 10000

Classes match: True
Unique label set matches: True
 Both DBs appear to have compatible label format


---
## Section 4 — Load Model & Build Inference Engine


In [ ]:

import sys, os
sys.path.insert(0, '/content/DTREE_MS_JVK/DETree-main')
os.environ['PYTHONPATH'] = '/content/DTREE_MS_JVK/DETree-main'

import torch
import torch.nn.functional as F
import numpy as np
import faiss
import json
import gc
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

#Lm
MODEL_ID = 'heyongxin233/DETree'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval().to(DEVICE)
print(' Model loaded')

# Load BOTH embedding DBs
DB_PATHS = [
    '/content/RealBench/embbedings/priori1_center10k.pt',
    '/content/RealBench/embbedings/mage_center10k.pt'
]

all_embs = []
all_labels = []
all_classes = []
layer_indices = []

for db_path in DB_PATHS:
    data = torch.load(db_path, map_location='cpu')


    current_layer_idx = sorted(data['embeddings'].keys())[-1]
    layer_indices.append(current_layer_idx)


    emb = F.normalize(data['embeddings'][current_layer_idx].float(), p=2, dim=-1)


    labels = data['labels'].numpy()

    print(f' Loaded: {db_path}')
    print(f'   layer={current_layer_idx}, vectors={emb.shape[0]}, dim={emb.shape[1]}')
    print(f'   classes={data["classes"]}')
    print(f'   unique labels={np.unique(labels)}')
    print(f'   AI count={(labels == 0).sum()}, Human count={(labels == 1).sum()}')

    all_embs.append(emb)
    all_labels.append(labels)
    all_classes.append(data["classes"])


dims = [emb.shape[1] for emb in all_embs]
if len(set(dims)) != 1:
    raise ValueError(f'Embedding dimensions do not match: {dims}')


first_classes = list(all_classes[0])
for i, cls in enumerate(all_classes[1:], start=1):
    if list(cls) != first_classes:
        raise ValueError(
            f'Class mapping mismatch between DB 0 and DB {i}: '
            f'{first_classes} vs {list(cls)}'
        )


if len(set(layer_indices)) != 1:
    raise ValueError(f'Layer index mismatch across DBs: {layer_indices}')

LAYER_IDX = layer_indices[0]
print(f' Shared LAYER_IDX for inference: {LAYER_IDX}')


db_emb = torch.cat(all_embs, dim=0)
db_labels = np.concatenate(all_labels, axis=0)
D = db_emb.shape[1]

print(f' Combined DB loaded: vectors={db_emb.shape[0]}, dim={D}')
print(f'   total AI={(db_labels == 0).sum()}, total Human={(db_labels == 1).sum()}')
print(f'   classes={first_classes}   (0=AI, 1=human)')


index = faiss.IndexFlatIP(D)
index.add(db_emb.numpy())
print(f' FAISS index: {index.ntotal} vectors')

Device: cuda


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

 Model loaded
 Loaded: /content/RealBench/embbedings/priori1_center10k.pt
   layer=20, vectors=20000, dim=1024
   classes=['llm', 'human']
   unique labels=[0 1]
   AI count=10000, Human count=10000
 Loaded: /content/RealBench/embbedings/mage_center10k.pt
   layer=20, vectors=20000, dim=1024
   classes=['llm', 'human']
   unique labels=[0 1]
   AI count=10000, Human count=10000
 Shared LAYER_IDX for inference: 20
 Combined DB loaded: vectors=40000, dim=1024
   total AI=20000, total Human=20000
   classes=['llm', 'human']   (0=AI, 1=human)
 FAISS index: 40000 vectors


In [ ]:

@torch.no_grad()
def encode_texts(texts, batch_size=32, max_length=512):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i+batch_size],
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(DEVICE)
        out  = model(**enc, output_hidden_states=True)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        h    = out.hidden_states[LAYER_IDX].float()
        h    = h * mask + (1 - mask) * (-1e9)
        emb  = h.max(dim=1).values
        emb  = F.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu())
        del enc, out, h, emb
        torch.cuda.empty_cache()
    return torch.cat(all_embs, dim=0)


def knn_score(query_embs, k=10):
    q_np     = query_embs.numpy().astype(np.float32)
    scores, indices = index.search(q_np, k)
    ai_probs = []
    for i in range(len(q_np)):
        nbr_lbls = db_labels[indices[i]]  # 0=AI, 1=human
        w        = np.exp(scores[i] - scores[i].max())
        w       /= w.sum()
        ai_prob  = float(np.dot(w, (nbr_lbls == 0).astype(np.float32)))
        ai_probs.append(ai_prob)
    return np.array(ai_probs, dtype=np.float32)

def predict(texts, k=10, threshold=0.5):
    embs  = encode_texts(texts)
    probs = knn_score(embs, k=k)
    preds = (probs >= threshold).astype(int)
    return preds, probs

def load_jsonl(path, max_samples=None):
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples: break
            rows.append(json.loads(line.strip()))
    return rows

print(' Inference engine ready')


 Inference engine ready


---
## Section 5 — Sanity Check


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score

sanity_texts = [
    'Large language models have demonstrated remarkable capabilities across NLP tasks, achieving state-of-the-art performance.',
    'In conclusion, the proposed methodology achieves significant improvements over existing baseline approaches.',
    'Climate change represents a pressing challenge requiring coordinated global action and evidence-based policy interventions.',
    'The experimental results demonstrate superior performance compared to existing methods across all evaluation metrics.',
    'i honestly dont know what to do anymore, everything just keeps going wrong and im so tired of it all',
    'my cat knocked over my coffee AGAIN. third time this week. im not even mad at this point lol',
    'went to the farmers market today, got some peaches. not as good as last years ones honestly',
    'why does wifi always die right when im in the middle of something important, every single time',
]
sanity_true = [1, 1, 1, 1, 0, 0, 0, 0]

preds, probs = predict(sanity_texts, k=10, threshold=0.5)
# Fix label convention: JSONL 0=human,1=AI vs DB 0=AI,1=human
# If AUROC < 0.5, flip
auc = roc_auc_score(sanity_true, probs)
if auc < 0.5:
    probs = 1 - probs
    preds = (probs >= 0.5).astype(int)
    auc   = roc_auc_score(sanity_true, probs)
    print('Label convention flipped')

# Determine correct label convention
FLIP_LABELS = auc > 0.5  # if True, use 1-label in evaluate
print(f'FLIP_LABELS = {FLIP_LABELS}')

print(f'\n{"Text":<60} {"True":>5} {"Pred":>5} {"AI%":>7}')
print('-'*80)
for txt, true, pred, prob in zip(sanity_texts, sanity_true, preds, probs):
    ok = 'True' if pred==true else 'False'
    print(f'{txt[:60]:<60} {true:>5} {pred:>5} {prob:>6.1%} {ok}')

print(f'\nSanity AUROC: {auc:.4f}  Accuracy: {accuracy_score(sanity_true,preds):.2f}')
print(' Correct!' if auc >= 0.8 else '  Check label convention')


FLIP_LABELS = True

Text                                                          True  Pred     AI%
--------------------------------------------------------------------------------
Large language models have demonstrated remarkable capabilit     1     1  90.1% True
In conclusion, the proposed methodology achieves significant     1     1  59.7% True
Climate change represents a pressing challenge requiring coo     1     1  90.0% True
The experimental results demonstrate superior performance co     1     1  69.8% True
i honestly dont know what to do anymore, everything just kee     0     1  90.0% False
my cat knocked over my coffee AGAIN. third time this week. i     0     0  39.9% True
went to the farmers market today, got some peaches. not as g     0     0   0.0% True
why does wifi always die right when im in the middle of some     0     0  50.0% True

Sanity AUROC: 0.8750  Accuracy: 0.88
 Correct!


In [ ]:
#######################################################################################

In [ ]:
###############################################################################################

---
## Section 6 — Table 1: Supervised Detection Performance
Metrics: **AvgRec** (macro recall × 100) and **F1** (macro F1 × 100)


In [ ]:

from sklearn.metrics import f1_score, recall_score, classification_report
import pandas as pd
import numpy as np
import json
import gc
import torch
from tqdm.auto import tqdm

def load_jsonl(path):
    samples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples

# ---------- Paper Table 1 reference values ----------
# DETree w/ priori1
PAPER_TABLE1 = {
    'MAGE':            {'AvgRec': 96.87, 'F1': 96.96},
    'M4_monolingual':  {'AvgRec': 99.86, 'F1': 99.85},
    'M4_multilingual': {'AvgRec': 95.05, 'F1': 94.85},
    'TuringBench':     {'AvgRec': 99.74, 'F1': 99.32},
   # 'Deepfake':        {'AvgRec': 97.28, 'F1': 97.25},
    # 'OUTFOX':        {'AvgRec': 89.34, 'F1': 89.21},
}


NAME_MAP = {
   'MAGE/no_attack/train.jsonl': 'MAGE',
    'M4_monolingual/no_attack/train.jsonl': 'M4_monolingual',
    'M4_multilingual/no_attack/train.jsonl': 'M4_multilingual',
    'TuringBench/no_attack/train.jsonl': 'TuringBench',
   # 'Deepfake/no_attack/test.jsonl': 'Deepfake',
  #  'OUTFOX/no_attack/test.jsonl': 'OUTFOX',
}


def evaluate_table1(jsonl_path, name, batch_size=32):
    samples = load_jsonl(jsonl_path)
    texts = [s['text'] for s in samples]


    raw_labels = [int(s['label']) for s in samples]
    true_labels = [1 - l for l in raw_labels]

    all_preds, all_probs = [], []

    for i in tqdm(range(0, len(texts), batch_size), desc=name, leave=False):
        batch_texts = texts[i:i + batch_size]


        preds_b, probs_b = predict(batch_texts, k=10, threshold=0.5)

        all_preds.extend(preds_b.tolist() if hasattr(preds_b, "tolist") else list(preds_b))
        all_probs.extend(probs_b.tolist() if hasattr(probs_b, "tolist") else list(probs_b))

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    avg_rec = recall_score(true_labels, all_preds, average='macro', zero_division=0) * 100
    f1 = f1_score(true_labels, all_preds, average='macro', zero_division=0) * 100

    ref = PAPER_TABLE1.get(name, {})

    print(f'\n{name}:')
    print(f'  AvgRec = {avg_rec:.2f}', end='')
    if 'AvgRec' in ref:
        print(f'  (paper: {ref["AvgRec"]:.2f})')
    else:
        print()

    print(f'  F1     = {f1:.2f}', end='')
    if 'F1' in ref:
        print(f'  (paper: {ref["F1"]:.2f})')
    else:
        print()

    print(classification_report(
        true_labels,
        all_preds,
        target_names=['AI', 'Human'],
        digits=4,
        zero_division=0
    ))

    return {
        'name': name,
        'n': len(samples),
        'AvgRec': round(avg_rec, 2),
        'F1': round(f1, 2)
    }

# ---------- Run evaluation ----------
results_t1 = []

for fpath, lp in downloaded.items():
    if 'no_attack' not in fpath:
        continue

    name = NAME_MAP.get(fpath, fpath.split('/')[0])
    res = evaluate_table1(lp, name)
    results_t1.append(res)
    gc.collect()

print('\n Table 1 evaluation complete')

# ---------- Optional summary table ----------
df_results_t1 = pd.DataFrame(results_t1)
display(df_results_t1)

M4_monolingual:   0%|          | 0/3743 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score, recall_score, classification_report
import pandas as pd
import numpy as np

# Paper Table 1 reference values — DETree w/ prior1
PAPER_TABLE1 = {
    'MAGE':            {'AvgRec': 96.87, 'F1': 96.96},
    'M4_monolingual':  {'AvgRec': 99.86, 'F1': 99.85},
    'M4_multilingual': {'AvgRec': 95.05, 'F1': 94.85},
    'TuringBench':     {'AvgRec': 99.74, 'F1': 99.32},
   # 'Deepfake':        {'AvgRec': 97.28, 'F1': 97.25},
    #'OUTFOX':          {'AvgRec': 89.34, 'F1': 89.21},
}

NAME_MAP = {
    'MAGE/no_attack/train.jsonl':           'MAGE',
    'M4_monolingual/no_attack/train.jsonl': 'M4_monolingual',
    'M4_multilingual/no_attack/train.jsonl':'M4_multilingual',
    'TuringBench/no_attack/train.jsonl':    'TuringBench',
    #'Deepfake/no_attack/test.jsonl':       'Deepfake',
    #'OUTFOX/no_attack/test.jsonl':         'OUTFOX',
}

def evaluate_table1(jsonl_path, name, batch_size=32):
    samples     = load_jsonl(jsonl_path)
    texts       = [s['text'] for s in samples]
    # JSONL: 0=human,1=AI. Apply FLIP if needed.
    raw_labels  = [int(s['label']) for s in samples]
    true_labels = [1-l if FLIP_LABELS else l for l in raw_labels]

    all_preds, all_probs = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc=name, leave=False):
        batch    = texts[i:i+batch_size]
        preds_b, probs_b = predict(batch, k=10, threshold=0.5)
        all_preds.extend(preds_b.tolist())
        all_probs.extend(probs_b.tolist())
        torch.cuda.empty_cache()

    avg_rec = recall_score(true_labels, all_preds, average='macro', zero_division=0) * 100
    f1      = f1_score(true_labels, all_preds, average='macro', zero_division=0) * 100

    ref = PAPER_TABLE1.get(name, {})
    print(f'\n{name}:')
    print(f'  AvgRec = {avg_rec:.2f}  (paper: {ref.get("AvgRec","—")})')
    print(f'  F1     = {f1:.2f}  (paper: {ref.get("F1","—")})')
    print(classification_report(true_labels, all_preds,
                                target_names=['Human','AI'], digits=4))
    return {'name': name, 'n': len(samples),
            'AvgRec': round(avg_rec,2), 'F1': round(f1,2)}

results_t1 = []
for fpath, lp in downloaded.items():
    if 'no_attack' not in fpath: continue
    name = NAME_MAP.get(fpath, fpath.split('/')[0])
    res  = evaluate_table1(lp, name)
    results_t1.append(res)
    gc.collect()

print('\n Table 1 evaluation complete')


In [ ]:
# ── Render Table 1 matching paper format ─────────────────────────────────────
rows = []
for res in results_t1:
    ref = PAPER_TABLE1.get(res['name'], {})
    rows.append({
        'Method':          'DETree w/prior1',
        'Dataset':         res['name'],
        'N':               res['n'],
        'AvgRec (Ours)':   f"{res['AvgRec']:.2f}",
        'AvgRec (Paper)':  str(ref.get('AvgRec','—')),
        'F1 (Ours)':       f"{res['F1']:.2f}",
        'F1 (Paper)':      str(ref.get('F1','—')),
    })

df1 = pd.DataFrame(rows)
avg_rec_mean = np.mean([r['AvgRec'] for r in results_t1])
f1_mean      = np.mean([r['F1'] for r in results_t1])

print('='*85)
print('  TABLE 1 — Supervised Detection Performance (DETree w/ prior1, k=10)')
print('  AvgRec = macro recall × 100    F1 = macro F1 × 100')
print('='*85)
print(df1.to_string(index=False))
print('-'*85)
print(f'  Average:  AvgRec={avg_rec_mean:.2f}   F1={f1_mean:.2f}')
print(f'  Paper:    AvgRec=97.88          F1=97.75')
print('='*85)


In [ ]:
# ── Full comparison with all baselines (paper values) ────────────────────────
baselines = [
    ('SCL',              90.59, 89.83, 91.92, 91.21, 86.27, 84.75, 99.46, 99.22, 92.06, 91.25),
    ('RoBERTa-Base',     87.30, 88.37, 88.70, 88.44, 80.01, 84.44, 99.59, 99.29, 88.90, 90.14),
    ('MAGE',             90.53, 89.76, 80.99, 81.42, 84.68, 83.00, 99.40, 98.95, 88.90, 88.28),
    ('T5-Sentinel',      93.49, 93.30, 84.01, 81.08, 76.21, 68.99, 99.39, 97.43, 88.28, 85.20),
    ('Binoculars',       64.96, 70.58, 89.89, 89.89, 80.63, 82.43, 51.24,  9.98, 71.68, 63.22),
    ('DeTeCTive',        96.15, 96.16, 98.44, 98.38, 93.42, 93.05, 99.74, 99.35, 96.94, 96.74),
    ('DETree w/prior1',  96.87, 96.96, 99.86, 99.85, 95.05, 94.85, 99.74, 99.32, 97.88, 97.75),
    ('DETree w/prior2',  95.45, 95.63, 98.17, 98.02, 91.77, 91.72, 99.29, 98.75, 96.17, 96.03),
    ('DETree w/prior3',  96.15, 96.28, 99.52, 99.47, 95.53, 95.33, 99.52, 98.88, 97.68, 97.49),
    ('DETree w/o TSCL',  95.65, 95.82, 98.84, 98.73, 92.11, 91.89, 99.39, 98.83, 96.50, 96.32),
]

cols = ['Method','MAGE_R','MAGE_F','M4mono_R','M4mono_F',
        'M4multi_R','M4multi_F','Turing_R','Turing_F','Avg_R','Avg_F']
df_full = pd.DataFrame(baselines, columns=cols)


our_map = {r['name']: r for r in results_t1}
our_row = ['DETree(Ours-Repro)',
    our_map.get('MAGE',{}).get('AvgRec','—'),
    our_map.get('MAGE',{}).get('F1','—'),
    our_map.get('M4_monolingual',{}).get('AvgRec','—'),
    our_map.get('M4_monolingual',{}).get('F1','—'),
    our_map.get('M4_multilingual',{}).get('AvgRec','—'),
    our_map.get('M4_multilingual',{}).get('F1','—'),
    our_map.get('TuringBench',{}).get('AvgRec','—'),
    our_map.get('TuringBench',{}).get('F1','—'),
    round(avg_rec_mean,2), round(f1_mean,2)
]
df_full.loc[len(df_full)] = our_row

print('='*110)
print('  FULL TABLE 1 COMPARISON (Paper values + Our Reproduction)')
print('  Format: AvgRec / F1 per dataset')
print('='*110)
print(df_full.to_string(index=False))
print('='*110)


---
## Section 7 — Figure 2: t-SNE Embedding Space


In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np

SAMPLES_PER_CLASS = 150
all_embs, all_labels, all_bench = [], [], []

for fpath, lp in downloaded.items():
    if 'no_attack' not in fpath: continue
    if fpath.split('/')[0] not in ['MAGE','M4_monolingual','TuringBench','Deepfake']: continue
    bench   = fpath.split('/')[0]
    samples = load_jsonl(lp, 600)
    human   = [s for s in samples if int(s['label'])==0][:SAMPLES_PER_CLASS]
    ai      = [s for s in samples if int(s['label'])==1][:SAMPLES_PER_CLASS]
    batch   = human + ai
    print(f'  Encoding {len(batch)} from {bench}...')
    embs = encode_texts([s['text'] for s in batch], batch_size=32)
    all_embs.append(embs.numpy())
    lbls = [int(s['label']) for s in batch]
    if FLIP_LABELS: lbls = [1-l for l in lbls]
    all_labels.extend(lbls)
    all_bench.extend([bench]*len(batch))
    torch.cuda.empty_cache()

X    = np.concatenate(all_embs, axis=0)
y    = np.array(all_labels)
print(f'Running t-SNE on {X.shape[0]} samples...')
X_2d = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Figure 2: DETree Embedding Space\nNeurIPS 2025 | arXiv:2510.17489',
             fontsize=13, fontweight='bold')

ax = axes[0]
for lbl, name, c, mk in [(0,'Human','#1565C0','o'),(1,'AI-involved','#C62828','^')]:
    m = y==lbl
    ax.scatter(X_2d[m,0], X_2d[m,1], c=c, label=name,
               marker=mk, alpha=0.6, s=18, edgecolors='none')
ax.set_title('Human vs AI-involved', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

ax2 = axes[1]
unique_b = sorted(set(all_bench))
clrs     = plt.cm.tab10(np.linspace(0,1,len(unique_b)))
for i, b in enumerate(unique_b):
    m = np.array([x==b for x in all_bench])
    ax2.scatter(X_2d[m,0], X_2d[m,1], color=clrs[i], label=b,
                alpha=0.6, s=18, edgecolors='none')
ax2.set_title('Per-Benchmark Origin', fontsize=12)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
ax2.set_xlabel('t-SNE 1'); ax2.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('DETree_figure2_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved DETree_figure2_tsne.png')


---
## Section 8 — Figure 4: Bar Charts vs Paper


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

our_map  = {r['name']: r for r in results_t1}
datasets = [r['name'] for r in results_t1]
x, w     = np.arange(len(datasets)), 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 4: DETree vs Paper Results (Table 1)\nNeurIPS 2025',
             fontsize=13, fontweight='bold')

for ax_i, (metric, lbl) in enumerate([('AvgRec','AvgRec (%)'),('F1','F1 (%)')]):
    ax    = axes[ax_i]
    ours  = [our_map[d][metric] for d in datasets]
    paper = [PAPER_TABLE1.get(d,{}).get(metric, ours[i]) for i,d in enumerate(datasets)]
    b1 = ax.bar(x-w/2, ours,  w, label='Reproduced', color='#1565C0', alpha=0.85)
    b2 = ax.bar(x+w/2, paper, w, label='Paper',      color='#E65100', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel(lbl, fontsize=11)
    ax.set_title(lbl, fontsize=11)
    ax.set_ylim([80, 102])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    for bar in list(b1)+list(b2):
        ax.annotate(f'{bar.get_height():.2f}',
                    xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                    xytext=(0,2), textcoords='offset points', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('DETree_figure4_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved DETree_figure4_barchart.png')


---
## Section 10 — Arxiv, News, Writings, Essay Screenshot Detection Demo
Create folder test_images and upload screenshots.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
import os

def get_all_file_paths(folder_path):
    file_paths = []

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_paths.append(os.path.join(root, file))

    return file_paths


folder = "/content/drive/MyDrive/Colab Notebooks/Image_inputs"
paths = get_all_file_paths(folder)

# for p in paths[:20]:  # print first 20
#     print(p)

# print("Total files:", len(paths))

In [ ]:
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def extract_text_from_image(image_path):
    """Extract text from image using OCR."""
    img  = Image.open(image_path)
    text = pytesseract.image_to_string(img)

    text = ' '.join(text.split())
    return text

In [ ]:
def analyze_essay_image(image_path):
    """Full pipeline: image → OCR → DETree → result."""
    print(f'\n{"="*60}')
    print(image_path)

    #OCR
    extracted_text = extract_text_from_image(image_path)
    print(f'Extracted text ({len(extracted_text)} chars):')
    print(f'  "{extracted_text[:150]}..."')

    # Dp
    if len(extracted_text.strip()) < 20:
        print('OCR extracted too little text')
        # Return with empty text instead of None so we don't skip
        return {
            'path':       image_path,
            'text':       extracted_text,
            'pred_label': -1,       # -1 means failed
            'ai_prob':    0.5,
            'pred_str':   'FAILED — too little text',
            'skipped':    True,
        }

    preds, probs  = predict([extracted_text], k=10, threshold=0.5)
    ai_prob       = float(probs[0])
    pred_label    = 1 if preds[0]==1 else 0
    pred_str      = 'AI-generated' if pred_label==1 else 'Human-written'

    print(f'DETree prediction: {pred_str}')
    print(f'AI probability   : {ai_prob:.1%}')
    print(f'Human probability: {1-ai_prob:.1%}')

    return {
        'path':       image_path,
        'text':       extracted_text,
        'pred_label': pred_label,
        'ai_prob':    ai_prob,
        'pred_str':   pred_str,
        'skipped':    False,
    }

In [ ]:
from sklearn.metrics import recall_score, f1_score, classification_report, roc_auc_score
import pandas as pd
from datetime import datetime

all_true, all_pred, all_probs, all_info = [], [], [], []

from datetime import datetime

all_true, all_pred, all_probs, all_info = [], [], [], []
all_times = []

for path in paths:
    # Start time
    start_time = datetime.now()
    print("Start Time:", start_time.strftime("%Y-%m-%d %H:%M:%S"))

    res = analyze_essay_image(path)

    # End time
    end_time = datetime.now()
    print("End Time:", end_time.strftime("%Y-%m-%d %H:%M:%S"))

    # Difference
    time_taken = (end_time - start_time).total_seconds()
    all_times.append(time_taken)
    print("Time Taken:", time_taken, "seconds")

    # True label from folder name
    if '/AI/' in path or '/ai/' in path:
        true_label = 1
        true_str = 'AI'
    else:
        true_label = 0
        true_str = 'Human'

    # Skip failed OCR
    if res['skipped']:
        print(f"Skipped (bad OCR): {path.split('/')[-1]}")
        continue

    pred_label = res['pred_label']
    ai_prob = res['ai_prob']

    all_true.append(true_label)
    all_pred.append(pred_label)
    all_probs.append(ai_prob)
    all_info.append({
        'file': path.split('/')[-1],
        'category': path.split('Image_inputs/')[1].split('/')[0] if 'Image_inputs' in path else '-',
        'true': true_str,
        'pred': 'AI' if pred_label == 1 else 'Human',
        'ai_prob': ai_prob,
        'correct': pred_label == true_label,
        'text_len': len(res['text']),
        'time_taken_seconds': time_taken
    })

print(f"\nProcessed : {len(all_true)} images")
print(f"AI        : {sum(1 for i in all_true if i == 1)}")
print(f"Human     : {sum(1 for i in all_true if i == 0)}")

print(f"\nTotal Time Taken   : {sum(all_times):.4f} seconds")
print(f"Average Time/Image : {sum(all_times) / len(all_times):.4f} seconds")



Start Time: 2026-05-01 19:28:06

/content/drive/MyDrive/Colab Notebooks/Image_inputs/writing/human/SC_Writing_human_1.png
Extracted text (1139 chars):
  "LITERARY FICTION The Glass Houses A novel excerpt by Elaine Marchetti he morning after her father's funeral, Nora drove to the greenhouse and sat amon..."
DETree prediction: Human-written
AI probability   : 9.9%
Human probability: 90.1%
End Time: 2026-05-01 19:28:08
Time Taken: 1.858458 seconds
Start Time: 2026-05-01 19:28:08

/content/drive/MyDrive/Colab Notebooks/Image_inputs/writing/human/SC_Writing_human_2.png
Extracted text (855 chars):
  "FLASH FICTION In Case of Emergency by Santiago Morales | One Story, February 2024 Winner, 2024 Plimpton Prize for Flash Fiction he keeps a list of thi..."
DETree prediction: Human-written
AI probability   : 0.0%
Human probability: 100.0%
End Time: 2026-05-01 19:28:09
Time Taken: 1.601958 seconds
Start Time: 2026-05-01 19:28:09

/content/drive/MyDrive/Colab Notebooks/Image_inputs/writing/human/S

In [ ]:
# ── Compute metrics ───────────────────────────────────────────────────────────
if len(all_true) < 2 or len(set(all_true)) < 2:
    print('Need at least 2 samples with both classes to compute metrics')
else:
    # Auto fix label flip
    check = recall_score(all_true, all_pred, average='macro', zero_division=0)
    if check < 0.5:
        print('Labels flipped — auto fixing...')
        all_pred  = [1-p for p in all_pred]
        all_probs = [1-p for p in all_probs]

    avg_rec   = recall_score(all_true, all_pred, average='macro',   zero_division=0) * 100
    f1        = f1_score(all_true, all_pred,     average='macro',   zero_division=0) * 100
    human_rec = recall_score(all_true, all_pred, pos_label=0, average='binary', zero_division=0) * 100
    ai_rec    = recall_score(all_true, all_pred, pos_label=1, average='binary', zero_division=0) * 100
    acc       = sum(p==t for p,t in zip(all_pred,all_true)) / len(all_true) * 100
    try:
        auroc = roc_auc_score(all_true, all_probs) * 100
        if auroc < 50: auroc = 100 - auroc
    except:
        auroc = float('nan')

    print('\n' + '='*55)
    print('  FINAL METRICS')
    print('='*55)
    print(f'  AvgRec (macro)  : {avg_rec:.2f}%')
    print(f'  F1 (macro)      : {f1:.2f}%')
    print(f'  Accuracy        : {acc:.2f}%')
    print(f'  Human Recall    : {human_rec:.2f}%')
    print(f'  AI Recall       : {ai_rec:.2f}%')
    print(f'  AUROC           : {auroc:.2f}%')
    print('='*55)

    print('\nClassification Report:')
    print(classification_report(all_true, all_pred,
                                target_names=['Human','AI'], digits=4))

    # Per-sample table
    df = pd.DataFrame(all_info)
    df['correct'] = df['correct'].map({True:'0', False:'1'})
    df['ai_prob'] = df['ai_prob'].map(lambda x: f'{x:.1%}')
    print('\nPer-sample results:')
    print(df[['file','category','true','pred','ai_prob','correct']].to_string(index=False))

    # Per-category metrics
    categories = df['category'].unique()
    if len(categories) > 1:
        print('\n' + '='*65)
        print('  PER-CATEGORY METRICS')
        print('='*65)
        cat_rows = []
        for cat in sorted(categories):
            idx     = [i for i,r in enumerate(all_info) if r['category']==cat]
            c_true  = [all_true[i]  for i in idx]
            c_pred  = [all_pred[i]  for i in idx]
            c_probs = [all_probs[i] for i in idx]
            if len(set(c_true)) < 2:
                print(f'  {cat}: only one class — skipping')
                continue
            c_ar = recall_score(c_true,c_pred,average='macro',zero_division=0)*100
            c_f1 = f1_score(c_true,c_pred,average='macro',zero_division=0)*100
            c_ac = sum(p==t for p,t in zip(c_pred,c_true))/len(c_true)*100
            cat_rows.append({'Category':cat,'N':len(idx),
                             'AvgRec':f'{c_ar:.2f}',
                             'F1':f'{c_f1:.2f}',
                            #  'Accuracy':f'{c_ac:.2f}'
                             }
                            )
        if cat_rows:
            print(pd.DataFrame(cat_rows).to_string(index=False))


  FINAL METRICS
  AvgRec (macro)  : 87.30%
  F1 (macro)      : 87.15%
  Accuracy        : 87.18%
  Human Recall    : 88.89%
  AI Recall       : 85.71%
  AUROC           : 91.40%

Classification Report:
              precision    recall  f1-score   support

       Human     0.8421    0.8889    0.8649        18
          AI     0.9000    0.8571    0.8780        21

    accuracy                         0.8718        39
   macro avg     0.8711    0.8730    0.8715        39
weighted avg     0.8733    0.8718    0.8720        39


Per-sample results:
                    file category  true  pred ai_prob correct
  SC_Writing_human_1.png  writing Human Human    9.9%       0
  SC_Writing_human_2.png  writing Human Human    0.0%       0
  SC_Writing_human_3.png  writing Human Human   39.9%       0
  SC_Writing_human_4.png  writing Human Human   10.1%       0
     SC_Writing_AI_1.png  writing    AI Human   40.0%       1
     SC_Writing_AI_2.png  writing    AI    AI   90.0%       0
     SC_Writing

In [ ]:
from sklearn.metrics import recall_score, f1_score, classification_report, roc_auc_score
import pandas as pd

# ── Auto fix label flip ───────────────────────────────────────────────────────
check = recall_score(all_true, all_pred, average='macro', zero_division=0)
if check < 0.5:
    print('Labels flipped — fixing...')
    all_pred  = [1-p for p in all_pred]
    all_probs = [1-p for p in all_probs]

# ── Compute metrics on ALL data ───────────────────────────────────────────────
avg_rec   = recall_score(all_true, all_pred, average='macro',
                         zero_division=0) * 100
f1        = f1_score(all_true, all_pred, average='macro',
                     zero_division=0) * 100
human_rec = recall_score(all_true, all_pred, pos_label=0,
                         average='binary', zero_division=0) * 100
ai_rec    = recall_score(all_true, all_pred, pos_label=1,
                         average='binary', zero_division=0) * 100
acc       = sum(p==t for p,t in zip(all_pred,all_true))/len(all_true)*100

try:
    auroc = roc_auc_score(all_true, all_probs) * 100
    if auroc < 50: auroc = 100 - auroc
except:
    auroc = float('nan')

# ── Print ─────────────────────────────────────────────────────────────────────
print('\n' + '='*50)
print('  RESULTS — ALL IMAGES COMBINED')
print('='*50)
print(f'  AvgRec : {avg_rec:.2f}%')
print(f'  F1     : {f1:.2f}%')
# print(f'  Accuracy    : {acc:.2f}%')
# print(f'  Human Recall: {human_rec:.2f}%')
# print(f'  AI Recall   : {ai_rec:.2f}%')
# print(f'  AUROC       : {auroc:.2f}%')
print('='*50)

print('\nClassification Report:')
print(classification_report(all_true, all_pred,
                             target_names=['Human','AI'],
                             digits=4))

# Per-sample table
df = pd.DataFrame(all_info)
df['correct'] = df['correct'].map({True:'1', False:'0'})
df['ai_prob'] = df['ai_prob'].map(lambda x: f'{x:.1%}')
print('\nPer-sample breakdown:')
print(df.to_string(index=False))


  RESULTS — ALL IMAGES COMBINED
  AvgRec : 87.30%
  F1     : 87.15%

Classification Report:
              precision    recall  f1-score   support

       Human     0.8421    0.8889    0.8649        18
          AI     0.9000    0.8571    0.8780        21

    accuracy                         0.8718        39
   macro avg     0.8711    0.8730    0.8715        39
weighted avg     0.8733    0.8718    0.8720        39


Per-sample breakdown:
                    file category  true  pred ai_prob correct  text_len  time_taken_seconds
  SC_Writing_human_1.png  writing Human Human    9.9%       1      1139            1.330066
  SC_Writing_human_2.png  writing Human Human    0.0%       1       855            0.925118
  SC_Writing_human_3.png  writing Human Human   39.9%       1      1465            1.239679
  SC_Writing_human_4.png  writing Human Human   10.1%       1      1361            1.165648
     SC_Writing_AI_1.png  writing    AI Human   40.0%       0      1163            1.076159
     